In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

# ── CELL 1B: Import Evelora Brand Theme ───────────────────────────────────

# sys.path tells Python where to look for files to import
# We add the parent folder so Python can find our assets folder
import sys
sys.path.append("../")

# Now import our brand colors from the theme file we created
# This means we never have to type hex codes manually again
from assets.evelora_theme import COLORS, PALETTE, BACKGROUND

# Quick check to confirm it worked
print("Brand theme loaded.")
print(f"Gold color: {COLORS['gold']}")
print(f"Background: {BACKGROUND}")
print(f"Palette has {len(PALETTE)} colors")
print("Setup complete.")

Brand theme loaded.
Gold color: #C5AA6D
Background: #1a1a1a
Palette has 6 colors
Setup complete.


In [2]:
# Load the industry adoption dataset — your hero file
df_industry = pd.read_csv("E:/EVELORA/evelora-ai-adoption-index/data/raw/5_Adoption_By_Industry.csv", skiprows=2)

# Load business function dataset
df_function = pd.read_csv("E:/EVELORA/evelora-ai-adoption-index/data/raw/2_AI_By_Business_Function.csv", skiprows=2)

# Load KPIs
df_kpis = pd.read_csv("E:/EVELORA/evelora-ai-adoption-index/data/raw/7_Key_KPIs_Summary.csv", skiprows=2)

print("Industry dataset:")
print(df_industry.shape)
print(df_industry.columns.tolist())
print()
print("Function dataset:")
print(df_function.shape)
print(df_function.columns.tolist())

Industry dataset:
(16, 7)
['Industry Sector', '2022 Adoption %', '2023 Adoption %', 'Early 2024 %', 'Late 2024 %', '2025 %', 'Key Data Points & Source']

Function dataset:
(15, 7)
['Business Function', '2021', '2022', '2023', 'Early 2024', 'Late 2024', '2025']


In [3]:
# ── CELL 3: Clean & Explore the Industry Dataset ──────────────────────────

# Drop the last row which is usually a footnote or NaN row
# axis=0 means we're dropping a row (not a column)
df_industry = df_industry.dropna(subset=["Industry Sector"])

# Strip any accidental spaces from industry names
# .str.strip() removes leading and trailing whitespace
df_industry["Industry Sector"] = df_industry["Industry Sector"].str.strip()

# Convert adoption columns to numeric
# errors='coerce' turns anything that isn't a number into NaN instead of crashing
for col in ["2022 Adoption %", "2023 Adoption %", "Early 2024 %", "Late 2024 %", "2025 %"]:
    df_industry[col] = pd.to_numeric(df_industry[col], errors="coerce")

# Multiply by 100 if values are in decimal form (0.72 instead of 72)
# We check the max value — if it's less than 2, values are decimals
if df_industry["2025 %"].max() < 2:
    for col in ["2022 Adoption %", "2023 Adoption %", "Early 2024 %", "Late 2024 %", "2025 %"]:
        df_industry[col] = df_industry[col] * 100

# Preview the clean data
# .round(1) rounds to 1 decimal place for readability
print("Clean industry data:")
print(df_industry[["Industry Sector", "2023 Adoption %", "Late 2024 %", "2025 %"]].round(1))

Clean industry data:
                                      Industry Sector  2023 Adoption %  \
0                               Technology / Software             76.0   
1                           Financial Services / BFSI             61.0   
2                               Professional Services             57.0   
3                                 Healthcare & Pharma             42.0   
4                                     Media & Telecom             55.0   
5                             Retail & Consumer Goods             45.0   
6                                  Energy & Materials             40.0   
7                                       Manufacturing             46.0   
8                                           Education             30.0   
9                                 Government & Public             28.0   
10                                        Agriculture             20.0   
11                           Logistics & Supply Chain             38.0   
12               

In [4]:
# ── CELL 4: Final Clean + Sort for Visualisation ──────────────────────────

# Drop row 15 which is a colour legend footnote, not real data
# We drop by index position using .drop() with the index label
df_industry = df_industry.drop(index=15).reset_index(drop=True)

# Sort industries by 2025 adoption % in descending order
# This makes our bar chart read from highest to lowest — much cleaner visually
df_industry_sorted = df_industry.sort_values("2025 %", ascending=False).reset_index(drop=True)

# Create a simple "gap" column — how much each industry grew from 2023 to 2025
# This tells us who moved fast and who barely budged
df_industry_sorted["Growth 2023 to 2025"] = (
    df_industry_sorted["2025 %"] - df_industry_sorted["2023 Adoption %"]
).round(1)

# Preview the final sorted table
print("Industries ranked by 2025 AI adoption:")
print(df_industry_sorted[["Industry Sector", "2023 Adoption %", "2025 %", "Growth 2023 to 2025"]].to_string())

Industries ranked by 2025 AI adoption:
              Industry Sector  2023 Adoption %  2025 %  Growth 2023 to 2025
0       Technology / Software             76.0    92.0                 16.0
1       Professional Services             57.0    82.0                 25.0
2   Financial Services / BFSI             61.0    80.0                 19.0
3             Media & Telecom             55.0    74.0                 19.0
4     Retail & Consumer Goods             45.0    68.0                 23.0
5          Energy & Materials             40.0    66.0                 26.0
6         Healthcare & Pharma             42.0    65.0                 23.0
7               Manufacturing             46.0    65.0                 19.0
8    Logistics & Supply Chain             38.0    63.0                 25.0
9                   Education             30.0    58.0                 28.0
10             Legal Services             22.0    52.0                 30.0
11        Government & Public             28.0   

In [5]:
# ── CELL 5: Hero Chart — AI Adoption by Industry 2025 ─────────────────────
# What this does: creates a horizontal bar chart showing how many companies
# in each industry report using AI. Higher bar = more AI adoption.

# Sort lowest to highest so the best industry appears at the TOP
df_plot = df_industry_sorted.sort_values("2025 %", ascending=True)

# Create the horizontal bar chart using plotly
fig = px.bar(
    df_plot,
    x="2025 %",           # bar length = adoption rate number
    y="Industry Sector",  # each row = one industry
    orientation="h",      # h = horizontal (easier to read long names)
    color="2025 %",       # color each bar based on its value
    color_continuous_scale=[
        [0.0, COLORS["dark"]],   # low adoption = dark brown
        [0.5, COLORS["gold"]],   # mid adoption = gold
        [1.0, COLORS["cream"]],  # high adoption = cream
    ],
)

# Style the overall chart layout
fig.update_layout(
    paper_bgcolor=BACKGROUND,      # outer background = Evelora black
    plot_bgcolor=BACKGROUND,       # inner chart background = Evelora black
    font=dict(
        color=COLORS["cream"],     # all text in cream
        size=13,
        family="Arial"
    ),
    coloraxis_showscale=False,     # hide the color legend (not needed)
    height=700,                    # tall enough to show all 15 industries
    width=900,                     # wide enough so nothing gets cut off
    margin=dict(l=10, r=120, t=130, b=60),
    xaxis_title="% of companies in this industry that report using AI (2025)",
    yaxis_title="",
    title=dict(
        text="Which industries are actually using AI in 2025?",
        font=dict(color=COLORS["gold"], size=18),
        x=0.5,              # center horizontally
        xanchor='center'    # anchor the title at its center
    ),
)

# Add a plain English subtitle so anyone can understand the chart
fig.add_annotation(
    text="Each bar shows what % of companies in that industry report using AI."
         " Higher = more adoption. Lower = falling behind.",
    xref="paper", yref="paper",
    x=0, y=1.08,
    showarrow=False,
    font=dict(color=COLORS["blush"], size=11),
    align="left",
)

# Show the % number at the end of each bar
fig.update_traces(
    texttemplate="%{x}%",
    textposition="outside",
    textfont=dict(color=COLORS["cream"], size=12),
)

# Style x axis (the numbers along the bottom)
fig.update_xaxes(
    range=[0, 110],
    gridcolor="#2a2a2a",
    color=COLORS["cream"],
)

# Style y axis (the industry names on the left)
fig.update_yaxes(
    color=COLORS["cream"],
    gridcolor=BACKGROUND,
)

fig.show()

# ── Plain English Summary ──────────────────────────────────────────────────
# This prints a simple explanation of what the chart is telling us

avg = df_plot["2025 %"].mean()

print("What this chart tells us:")
print(f"Highest adopter : {df_plot.iloc[-1]['Industry Sector']} at {df_plot.iloc[-1]['2025 %']}%")
print(f"Lowest adopter  : {df_plot.iloc[0]['Industry Sector']} at {df_plot.iloc[0]['2025 %']}%")
print(f"Industry average: {avg:.1f}%")
print()
print("Above average (genuinely adopting AI):")
above = df_plot[df_plot["2025 %"] > avg]
print(above[["Industry Sector", "2025 %"]].to_string(index=False))
print()
print("Below average (talking about AI more than doing it):")
below = df_plot[df_plot["2025 %"] < avg]
print(below[["Industry Sector", "2025 %"]].to_string(index=False))

# ── Save as HTML for LinkedIn screenshot ───────────────────────────────────
# write_html saves the chart as a webpage file
# Open it in Chrome, screenshot it, and post to LinkedIn

fig.write_html("../assets/chart1_industry_adoption.html")
print()
print("Chart saved. Open assets/chart1_industry_adoption.html in Chrome to screenshot.")

What this chart tells us:
Highest adopter : Technology / Software at 92.0%
Lowest adopter  : Agriculture at 42.0%
Industry average: 63.1%

Above average (genuinely adopting AI):
          Industry Sector  2025 %
      Healthcare & Pharma    65.0
            Manufacturing    65.0
       Energy & Materials    66.0
  Retail & Consumer Goods    68.0
          Media & Telecom    74.0
Financial Services / BFSI    80.0
    Professional Services    82.0
    Technology / Software    92.0

Below average (talking about AI more than doing it):
         Industry Sector  2025 %
             Agriculture    42.0
   Environment / Climate    44.0
    Real Estate & Const.    46.0
     Government & Public    50.0
          Legal Services    52.0
               Education    58.0
Logistics & Supply Chain    63.0

Chart saved. Open assets/chart1_industry_adoption.html in Chrome to screenshot.


In [6]:
# The chart shows 15 industries ranked by how many companies in each sector actually report using AI. The average across all industries is 63.1%. Anything above that line is genuinely adopting AI. Anything below is falling behind — regardless of how much they talk about it.
# The 3 insights your data just revealed that will shock people:

# Agriculture talks about "AI transforming farming" constantly but only 42% of companies use it. Dead last.
# Environment and Climate — every conference says "AI will save the planet" but it's second to last at 44%.
# Legal Services grew the FASTEST of any industry — 30 percentage points in 2 years — and nobody is talking about it.

In [7]:
# ── CELL 6: Chart 2 — Who Grew the Fastest? ───────────────────────────────

import plotly.graph_objects as go

# ── 1. Prep data ───────────────────────────────────────────────────────────

df_growth = (df_industry_sorted
             .sort_values("Growth 2023 to 2025", ascending=True)
             .dropna(subset=["Growth 2023 to 2025"])
             .reset_index(drop=True))

industries = df_growth["Industry Sector"].tolist()
val_2023   = df_growth["2023 Adoption %"].tolist()
val_2025   = df_growth["2025 %"].tolist()
growth     = df_growth["Growth 2023 to 2025"].tolist()
n          = len(industries)

fastest_name = df_growth.iloc[-1]["Industry Sector"]
fastest_pts  = int(df_growth.iloc[-1]["Growth 2023 to 2025"])
slowest_name = df_growth.iloc[0]["Industry Sector"]
slowest_pts  = int(df_growth.iloc[0]["Growth 2023 to 2025"])
avg_pts      = round(df_growth["Growth 2023 to 2025"].mean(), 1)

# ── 2. Figure ──────────────────────────────────────────────────────────────

fig2 = go.Figure()

# Connecting lines between dots
for i in range(n):
    fig2.add_shape(type="line",
                   x0=val_2023[i], x1=val_2025[i], y0=i, y1=i,
                   line=dict(color=COLORS["dark"], width=2.5))

# 2023 dots
fig2.add_trace(go.Scatter(
    x=val_2023, y=list(range(n)),
    mode="markers+text", name="2023 — where they started",
    marker=dict(color=COLORS["dark"], size=10, line=dict(color=COLORS["blush"], width=1.2)),
    text=[f"{v:.0f}%" for v in val_2023], textposition="middle left",
    textfont=dict(color=COLORS["blush"], size=10),
    hovertemplate="<b>%{customdata}</b><br>2023: %{x:.0f}%<extra></extra>",
    customdata=industries,
))

# 2025 dots
fig2.add_trace(go.Scatter(
    x=val_2025, y=list(range(n)),
    mode="markers+text", name="2025 — where they are now",
    marker=dict(color=COLORS["gold"], size=10, line=dict(color=COLORS["cream"], width=1.2)),
    text=[f"+{g:.0f} pts → {v:.0f}%" for g, v in zip(growth, val_2025)],
    textposition="middle right",
    textfont=dict(color=COLORS["gold"], size=10),
    hovertemplate="<b>%{customdata}</b><br>2025: %{x:.0f}%<extra></extra>",
    customdata=industries,
))

# ── 3. Layout ──────────────────────────────────────────────────────────────

fig2.update_layout(
    paper_bgcolor=BACKGROUND,
    plot_bgcolor=BACKGROUND,
    font=dict(color=COLORS["cream"], size=12, family="Arial"),
    height=max(620, n * 42 + 260),
    width=1100,
    # b=200 gives x-axis title + insight box room to live without touching
    margin=dict(l=210, r=180, t=185, b=200),

    title=dict(
        text="The AI race is on — and the winners might surprise you (2023 → 2025)",
        font=dict(color=COLORS["gold"], size=19),
        x=0.5, xanchor="center", y=0.988,
    ),

    legend=dict(
        bgcolor=BACKGROUND, bordercolor=COLORS["dark"], borderwidth=1,
        font=dict(color=COLORS["cream"], size=11),
        orientation="v", x=1.02, xanchor="left", y=0.5, yanchor="middle",
    ),

    xaxis=dict(
        title=dict(text="AI adoption rate (%)", font=dict(color=COLORS["cream"], size=12)),
        color=COLORS["cream"], gridcolor="#2e2720",
        zeroline=False, showline=False, ticksuffix="%", range=[0, 122],
    ),

    yaxis=dict(
        tickvals=list(range(n)), ticktext=industries,
        color=COLORS["cream"], gridcolor="#2e2720", tickfont=dict(size=11),
        # Bottom dot starts at y=0, top dot at y=n-1.
        # range bottom = -1.2 pushes ALL dots noticeably up from the x-axis border.
        # range top   =  n + 0.2 gives a small ceiling gap.
        range=[-1.2, n + 0.2],
    ),

    hovermode="closest",
)

# ── 4. KPI cards ──────────────────────────────────────────────────────────

card_specs = [
    (0.00, 0.33, "FASTEST MOVER", fastest_name,     f"jumped {fastest_pts} points", COLORS["gold"]),
    (0.33, 0.66, "SLOWEST MOVER", slowest_name,     f"only {slowest_pts} points",   COLORS["blush"]),
    (0.66, 1.00, "AVG. GROWTH",   f"{avg_pts} pts", "across all industries",        COLORS["cream"]),
]

for x0, x1, label, value, sub, val_col in card_specs:
    xc = (x0 + x1) / 2
    fig2.add_shape(type="rect", xref="paper", yref="paper",
                   x0=x0+0.005, x1=x1-0.005, y0=1.055, y1=1.20,
                   fillcolor="#252220", line=dict(color=COLORS["dark"], width=1), layer="below")
    fig2.add_annotation(xref="paper", yref="paper", x=xc, y=1.182,
                        text=label, showarrow=False, xanchor="center",
                        font=dict(color=COLORS["dark"], size=9))
    fig2.add_annotation(xref="paper", yref="paper", x=xc, y=1.138,
                        text=f"<b>{value}</b>", showarrow=False, xanchor="center",
                        font=dict(color=val_col, size=13))
    fig2.add_annotation(xref="paper", yref="paper", x=xc, y=1.078,
                        text=sub, showarrow=False, xanchor="center",
                        font=dict(color=COLORS["blush"], size=10))

# ── 5. Insight box ─────────────────────────────────────────────────────────
# Sits in the bottom margin (below the x-axis title).
# Two short lines instead of one long line — guaranteed to stay inside the box.

line1 = f"{fastest_name} leads the pack, but Education and Environment/Climate are right behind."
line2 = "Turns out the 'boring' industries are quietly going all-in on AI."

# Box boundaries in paper coords (below 0 = below the plot area)
BOX_TOP = -0.10
BOX_BOT = -0.38

fig2.add_shape(type="rect", xref="paper", yref="paper",
               x0=0.02, x1=0.98,
               y0=BOX_BOT, y1=BOX_TOP,
               fillcolor="#1f1a16",
               line=dict(color=COLORS["dark"], width=0.5), layer="below")

# Gold left accent bar
fig2.add_shape(type="line", xref="paper", yref="paper",
               x0=0.02, x1=0.02, y0=BOX_BOT, y1=BOX_TOP,
               line=dict(color=COLORS["gold"], width=4))

# Title of insight box
fig2.add_annotation(xref="paper", yref="paper",
                    x=0.04, y=BOX_TOP - 0.04,
                    text="<b>The surprising part</b>",
                    showarrow=False, xanchor="left",
                    font=dict(color=COLORS["cream"], size=11))

# Line 1 of insight
fig2.add_annotation(xref="paper", yref="paper",
                    x=0.04, y=BOX_TOP - 0.14,
                    text=line1,
                    showarrow=False, xanchor="left",
                    font=dict(color=COLORS["blush"], size=10))

# Line 2 of insight — separate annotation keeps it inside the box
fig2.add_annotation(xref="paper", yref="paper",
                    x=0.04, y=BOX_TOP - 0.24,
                    text=line2,
                    showarrow=False, xanchor="left",
                    font=dict(color=COLORS["blush"], size=10))

fig2.show()

# ── 6. Summary ────────────────────────────────────────────────────────────

print(f"Fastest: {fastest_name} +{fastest_pts} pts  |  "
      f"Slowest: {slowest_name} +{slowest_pts} pts  |  Avg: {avg_pts} pts")
fig2.write_html("../assets/chart2_growth.html")
print("Chart 2 saved.")

Fastest: Legal Services +30 pts  |  Slowest: Technology / Software +16 pts  |  Avg: 23.0 pts
Chart 2 saved.


In [8]:
# ── CELL 7: Chart 3 — AI Adoption by Business Function ────────────────────
# What this does: shows which PARTS of a company are using AI
# For example: is the IT team using AI? What about HR? Marketing?
# This is interesting because it shows WHERE inside companies AI is actually happening

# Clean the business function dataset first
# Drop the last row which is a color legend note, not real data
df_function = df_function.dropna(subset=["Business Function"])
df_function = df_function[~df_function["Business Function"].str.contains("Color", na=False)]

# Convert the 2025 column to numbers
# errors="coerce" turns anything non-numeric into NaN instead of crashing
df_function["2025"] = pd.to_numeric(df_function["2025"], errors="coerce")

# Multiply by 100 if values are in decimal form (0.43 instead of 43)
if df_function["2025"].max() < 2:
    df_function["2025"] = df_function["2025"] * 100

# Round to 1 decimal place for cleaner display
df_function["2025"] = df_function["2025"].round(1)

# Drop any rows where 2025 value is missing
df_function = df_function.dropna(subset=["2025"])

# Sort lowest to highest so highest appears at top of chart
df_func_sorted = df_function.sort_values("2025", ascending=True)

# Preview so you can see what we are working with
print("Business functions ranked by 2025 AI adoption:")
print(df_func_sorted[["Business Function", "2025"]].to_string(index=False))

Business functions ranked by 2025 AI adoption:
        Business Function  2025
  Strategy & Corp Finance  18.0
          Human Resources  21.0
            R&D / Science  22.0
 Supply Chain & Inventory  23.0
           Risk & Finance  24.0
        Manufacturing/Ops  27.0
     Knowledge Management  28.0
       Service Operations  33.0
    Product & Service Dev  34.0
        Marketing & Sales  36.0
IT & Software Engineering  43.0


In [9]:
# IT & Software Engineering is at 43% — the highest. That makes sense, tech people adopt tech first.
# But look at the bottom: Strategy & Corp Finance at just 18%. 
# These are the executives making decisions about AI budget, and only 18% of them are actually using AI themselves. 
# HR is at 21%. The people who hire for AI roles barely use AI in their own work.

In [10]:
# ── CELL 8: Chart 3 — Business Function Chart ─────────────────────────────
# What this shows: which department inside a company uses AI the most
# Shocking finding: the executives making AI decisions use it the least

fig3 = px.bar(
    df_func_sorted,
    x="2025",                  # bar length = adoption rate
    y="Business Function",     # each bar = one department
    orientation="h",
    color="2025",
    color_continuous_scale=[
        [0.0, COLORS["dark"]],
        [0.5, COLORS["gold"]],
        [1.0, COLORS["cream"]],
    ],
)

fig3.update_layout(
    paper_bgcolor=BACKGROUND,
    plot_bgcolor=BACKGROUND,
    font=dict(color=COLORS["cream"], size=13, family="Arial"),
    coloraxis_showscale=False,
    height=650,
    width=1100,
    margin=dict(l=200, r=150, t=160, b=80),
    xaxis_title="% of companies using AI in this business function (2025)",
    yaxis_title="",
    title=dict(
        text="Where inside companies is AI actually being used? (2025)",
        font=dict(color=COLORS["gold"], size=20),
        x=0.5,
        xanchor="center",
        y=0.97,
    ),
)

# Centered subtitle
fig3.add_annotation(
    text="Each bar = % of companies where this department is actively using AI."
         " The gap between IT and HR will surprise you.",
    xref="paper", yref="paper",
    x=0.5,
    y=1.07,
    showarrow=False,
    font=dict(color=COLORS["blush"], size=11),
    align="center",
)

# Show % at end of each bar
fig3.update_traces(
    texttemplate="%{x}%",
    textposition="outside",
    textfont=dict(color=COLORS["cream"], size=12),
)

fig3.update_xaxes(
    range=[0, 55],
    gridcolor="#2a2a2a",
    color=COLORS["cream"],
)

fig3.update_yaxes(
    color=COLORS["cream"],
    gridcolor=BACKGROUND,
)

fig3.show()

# Plain English summary
print("What this chart tells us:")
print(f"Highest: {df_func_sorted.iloc[-1]['Business Function']} at {df_func_sorted.iloc[-1]['2025']}%")
print(f"Lowest : {df_func_sorted.iloc[0]['Business Function']} at {df_func_sorted.iloc[0]['2025']}%")
print()
print("The uncomfortable truth:")
print("The executives making AI investment decisions use AI the least.")
print("Strategy and Finance: 18%. HR: 21%. IT: 43%.")

# Save
fig3.write_html("../assets/chart3_business_function.html")
print()
print("Chart 3 saved.")

What this chart tells us:
Highest: IT & Software Engineering at 43.0%
Lowest : Strategy & Corp Finance at 18.0%

The uncomfortable truth:
The executives making AI investment decisions use AI the least.
Strategy and Finance: 18%. HR: 21%. IT: 43%.

Chart 3 saved.


In [11]:
# Most companies say they use AI. But "using AI" means very different things. 
# Tech companies are genuinely transformed at 92%. Agriculture and Environment are at 42% and 44%, the exact sectors everyone says AI will revolutionize. 
# And the most uncomfortable finding: the executives making decisions about AI budgets are the least likely to use AI themselves. 
# Strategy and Finance at 18%. HR at 21%. While IT quietly runs at 43%.

In [12]:
# Quick check — make sure our cleaned data is still available
# .shape tells us rows x columns
# If this errors, we need to re-run the earlier cells first

print(f"Rows: {df_industry_sorted.shape[0]}")
print(f"Columns: {df_industry_sorted.shape[1]}")
print(f"Columns available: {df_industry_sorted.columns.tolist()}")

Rows: 15
Columns: 8
Columns available: ['Industry Sector', '2022 Adoption %', '2023 Adoption %', 'Early 2024 %', 'Late 2024 %', '2025 %', 'Key Data Points & Source', 'Growth 2023 to 2025']


In [13]:
# Every industry gets 3 mini-scores:
# Score A — Current Adoption (out of 50 points)
# The industry with the highest adoption (92%) gets 50 points. The lowest (42%) gets 0. Everyone else gets scored proportionally in between. This is called normalisation — we scale everyone between 0 and 50.
# Score B — Growth Speed (out of 30 points)
# The industry that grew the fastest (Legal Services, 30 points) gets 30 points. The slowest (Tech, 16 points) gets 0. Scaled in between.
# Score C — Consistency (out of 20 points)
# How smooth was their growth from 2022 to 2025? An industry that grew steadily every year scores higher than one that jumped randomly. We measure this using the difference between Early 2024 and Late 2024 — small difference means consistent growth.
# Add A + B + C = final score out of 100.

In [14]:
# ── CELL 10: AI Readiness Score ────────────────────────────────────────────
# What this does: gives every industry a score out of 100
# Think of it like a report card for AI adoption
# Score = how much they use AI + how fast they grew + how consistently they grew

# We work on a copy so we never damage our original cleaned data
df_score = df_industry_sorted.copy()

# ── Score A: Current Adoption (worth 50 points) ───────────────────────────
# Formula: (industry value - minimum value) / (maximum - minimum) * 50
# This scales every industry between 0 and 50 based on their 2025 adoption
# The best industry gets 50, the worst gets 0, others fall in between

min_adopt = df_score["2025 %"].min()   # lowest adoption in dataset
max_adopt = df_score["2025 %"].max()   # highest adoption in dataset

df_score["Score A"] = (
    (df_score["2025 %"] - min_adopt) / (max_adopt - min_adopt) * 50
).round(1)

# ── Score B: Growth Speed (worth 30 points) ───────────────────────────────
# Same formula but using the Growth column we already created
# Fast grower = closer to 30 points

min_growth = df_score["Growth 2023 to 2025"].min()
max_growth = df_score["Growth 2023 to 2025"].max()

df_score["Score B"] = (
    (df_score["Growth 2023 to 2025"] - min_growth) / (max_growth - min_growth) * 30
).round(1)

# ── Score C: Consistency (worth 20 points) ────────────────────────────────
# We measure consistency by looking at the gap between Early 2024 and Late 2024
# Small gap = grew steadily = more consistent = higher score
# Large gap = erratic growth = lower score

df_score["Gap"] = abs(df_score["Late 2024 %"] - df_score["Early 2024 %"])

min_gap = df_score["Gap"].min()
max_gap = df_score["Gap"].max()

# Note: we REVERSE this score because small gap = good consistency
# So we subtract from max_gap before scaling
df_score["Score C"] = (
    (max_gap - df_score["Gap"]) / (max_gap - min_gap) * 20
).round(1)

# ── Final Score: Add all three together ───────────────────────────────────
df_score["AI Readiness Score"] = (
    df_score["Score A"] + df_score["Score B"] + df_score["Score C"]
).round(1)

# ── Show the results sorted by final score ────────────────────────────────
df_final = df_score[[
    "Industry Sector",
    "2025 %",
    "Growth 2023 to 2025",
    "Score A",
    "Score B", 
    "Score C",
    "AI Readiness Score"
]].sort_values("AI Readiness Score", ascending=False).reset_index(drop=True)

print("AI Readiness Scores — Industries Ranked:")
print()
print(df_final.to_string(index=False))

AI Readiness Scores — Industries Ranked:

          Industry Sector  2025 %  Growth 2023 to 2025  Score A  Score B  Score C  AI Readiness Score
    Professional Services    82.0                 25.0     40.0     19.3     15.0                74.3
    Technology / Software    92.0                 16.0     50.0      0.0     15.0                65.0
       Energy & Materials    66.0                 26.0     24.0     21.4     15.0                60.4
Financial Services / BFSI    80.0                 19.0     38.0      6.4     15.0                59.4
          Media & Telecom    74.0                 19.0     32.0      6.4     20.0                58.4
  Retail & Consumer Goods    68.0                 23.0     26.0     15.0     15.0                56.0
 Logistics & Supply Chain    63.0                 25.0     21.0     19.3     10.0                50.3
                Education    58.0                 28.0     16.0     25.7      5.0                46.7
            Manufacturing    65.0       

In [15]:
# Professional Services scores 74.3 and ranks 
# FIRST — not Technology. Why? Because Tech grew the slowest (Score B = 0.0) despite having the highest current adoption. Professional Services grew fast AND consistently. 
# That's the insight no raw adoption chart would show you.Legal Services scores only 40 despite being the fastest grower — because its current adoption is still low at 52%. 
# Fast but starting from behind.Real Estate scores lowest at 21.9. Low adoption, slow growth, inconsistent. The most AI-unprepared industry.

In [16]:
# ── CELL 11 (REBUILT): AI Readiness Score — Bubble Chart ──────────────────
# Why a bubble chart: each industry becomes a circle
# Circle size = AI Readiness Score (bigger = more ready)
# X axis = current adoption in 2025
# Y axis = growth speed from 2023 to 2025
# This lets us see THREE things at once in one visual

import plotly.graph_objects as go

# Color each bubble based on readiness level
df_chart = df_final.copy()
df_chart["Readiness Level"] = pd.cut(
    df_chart["AI Readiness Score"],
    bins=[-1, 40, 60, 100],
    labels=["Lagging Behind", "Developing", "High Readiness"],
)

bubble_colors = df_chart["Readiness Level"].map({
    "High Readiness": COLORS["gold"],
    "Developing": COLORS["blush"],
    "Lagging Behind": COLORS["dark"],
})

fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x=df_chart["2025 %"],                    # x position = current adoption
    y=df_chart["Growth 2023 to 2025"],        # y position = how fast it grew
    mode="markers+text",                      # show circles AND text labels
    marker=dict(
        size=df_chart["AI Readiness Score"],  # bubble size = readiness score
        color=bubble_colors,                  # bubble color = readiness level
        opacity=0.85,
        line=dict(color=COLORS["cream"], width=1),  # thin cream border on each bubble
    ),
    text=df_chart["Industry Sector"],         # label inside/near each bubble
    textposition="top center",
    textfont=dict(color=COLORS["cream"], size=10),

    # What shows when you hover over a bubble
    customdata=df_chart[["AI Readiness Score", "Readiness Level"]].values,
    hovertemplate=(
        "<b>%{text}</b><br>"
        "2025 Adoption: %{x}%<br>"
        "Growth: %{y} pts<br>"
        "AI Readiness Score: %{customdata[0]}<br>"
        "Level: %{customdata[1]}"
        "<extra></extra>"
    ),
))

fig4.update_layout(
    paper_bgcolor=BACKGROUND,
    plot_bgcolor="#111111",
    font=dict(color=COLORS["cream"], size=12, family="Arial"),
    height=700,
    width=1100,
    margin=dict(l=80, r=80, t=180, b=80),
    xaxis_title="Current AI Adoption in 2025 (%)",
    yaxis_title="Growth in AI Adoption (2023 to 2025, percentage points)",
    title=dict(
        text="AI Readiness Score — Industry Bubble Map (2025)",
        font=dict(color=COLORS["gold"], size=20),
        x=0.5,
        xanchor="center",
        y=0.97,
    ),
)

# Subtitle
fig4.add_annotation(
    text="Bubble size = AI Readiness Score out of 100."
         " Right = high adoption. Up = fast growth. Bigger = more ready."
         " Hover over any bubble for full details.",
    xref="paper", yref="paper",
    x=0.5, y=1.10,
    showarrow=False,
    font=dict(color=COLORS["blush"], size=11),
    align="center",
)

# Add quadrant lines to divide chart into 4 zones
avg_x = df_chart["2025 %"].mean()
avg_y = df_chart["Growth 2023 to 2025"].mean()

# Vertical line at average adoption
fig4.add_vline(
    x=avg_x,
    line_dash="dash",
    line_color="#333333",
    line_width=1,
)

# Horizontal line at average growth
fig4.add_hline(
    y=avg_y,
    line_dash="dash",
    line_color="#333333",
    line_width=1,
)

# Label the 4 quadrants
fig4.add_annotation(
    x=df_chart["2025 %"].max(), y=df_chart["Growth 2023 to 2025"].max(),
    text="High adoption + Fast growth",
    showarrow=False,
    font=dict(color=COLORS["gold"], size=10),
    xanchor="right",
)
fig4.add_annotation(
    x=df_chart["2025 %"].min(), y=df_chart["Growth 2023 to 2025"].max(),
    text="Low adoption + Fast growth",
    showarrow=False,
    font=dict(color=COLORS["blush"], size=10),
    xanchor="left",
)
fig4.add_annotation(
    x=df_chart["2025 %"].max(), y=df_chart["Growth 2023 to 2025"].min(),
    text="High adoption + Slow growth",
    showarrow=False,
    font=dict(color=COLORS["blush"], size=10),
    xanchor="right",
)
fig4.add_annotation(
    x=df_chart["2025 %"].min(), y=df_chart["Growth 2023 to 2025"].min(),
    text="Low adoption + Slow growth",
    showarrow=False,
    font=dict(color=COLORS["dark"], size=10),
    xanchor="left",
)

fig4.update_xaxes(gridcolor="#1e1e1e", color=COLORS["cream"])
fig4.update_yaxes(gridcolor="#1e1e1e", color=COLORS["cream"])

fig4.show()

# Save
fig4.write_html("../assets/chart4_readiness_bubble.html")
print("Chart 4 saved.")

Chart 4 saved.


In [17]:
# Top right (High adoption + Fast growth) — nobody is here yet. This is the ideal zone. The closest is Professional Services which is almost there.
# Top left (Low adoption + Fast growth) — Legal Services and Education. These are the surprise risers. Low now but moving fast. Watch these industries in the next 2 years.
# Bottom right (High adoption + Slow growth) — Technology and Financial Services. They adopted early, now slowing down. Mature AI users.
# Bottom left (Low adoption + Slow growth) — Agriculture, Real Estate, Government. The genuinely left-behind industries. Small bubbles, wrong corner.
# The bubble sizes add a third layer — Professional Services has the biggest gold bubble. Tech has a decent sized gold bubble but it's in the slow growth zone. Legal Services has a medium pink bubble but it's racing upward.


In [18]:
# ── CELL 12: Predictive Model — When Will Each Industry Hit 80%? ───────────
# What this does: uses linear regression to predict the future
# Linear regression finds the growth trend of each industry
# Then extends that trend forward to find when adoption hits 80%

# We need these two tools from scikit-learn
# LinearRegression finds the trend line
# numpy helps us do the math
from sklearn.linear_model import LinearRegression
import numpy as np

# Our data points in time — we convert years to numbers
# 2022=0, 2023=1, Early 2024=1.5, Late 2024=2, 2025=2.5
# This is because the model needs evenly comparable numbers
time_points = np.array([0, 1, 1.5, 2, 2.5]).reshape(-1, 1)
# reshape(-1,1) means "make this a column" — sklearn needs this shape

# The columns in our dataset that match each time point
adoption_cols = [
    "2022 Adoption %",
    "2023 Adoption %",
    "Early 2024 %",
    "Late 2024 %",
    "2025 %"
]

# Store prediction results here
results = []

# Loop through every industry one by one
for _, row in df_score.iterrows():

    industry = row["Industry Sector"]

    # Get this industry's adoption values across all time points
    # This is like saying: here are the plant heights per week
    y = row[adoption_cols].values.astype(float).reshape(-1, 1)

    # Fit the linear regression model
    # This finds the best straight line through our data points
    model = LinearRegression()
    model.fit(time_points, y)

    # The slope tells us how fast this industry grows per time unit
    slope = model.coef_[0][0]

    # The intercept tells us where the line starts (value at time 0)
    intercept = model.intercept_[0]

    # Current adoption in 2025 (time point 2.5)
    current = row["2025 %"]

    # Only predict if industry hasn't already hit 80%
    if current >= 80:
        # Already there
        year_to_80 = "Already at 80%+"
        years_needed = 0
    elif slope <= 0:
        # Declining or flat trend — will never reach 80%
        year_to_80 = "Unlikely at current rate"
        years_needed = 999
    else:
        # Math: if y = slope * x + intercept, solve for x when y = 80
        # x = (80 - intercept) / slope
        # Then convert x back to a real year (2022 + x*2 years of data)
        x_to_80 = (80 - intercept) / slope
        years_from_2022 = x_to_80 * 2   # each unit = ~2 years of real time
        predicted_year = 2022 + years_from_2022
        predicted_year = round(predicted_year)
        year_to_80 = str(predicted_year)
        years_needed = predicted_year - 2025

    results.append({
        "Industry Sector"  : industry,
        "2025 Adoption %"  : current,
        "Growth Rate/yr"   : round(slope * 2, 1),  # convert to real yearly rate
        "Predicted Year to 80%" : year_to_80,
        "Years from Now"   : years_needed,
    })

# Convert results to a dataframe
df_predict = pd.DataFrame(results)

# Sort by years needed — fastest to hit 80% at the top
df_predict = df_predict.sort_values("Years from Now").reset_index(drop=True)

# Plain English output
print("Which industry will hit 80% AI adoption first?")
print()
print(df_predict.to_string(index=False))

Which industry will hit 80% AI adoption first?

          Industry Sector  2025 Adoption %  Growth Rate/yr Predicted Year to 80%  Years from Now
    Technology / Software             92.0            18.4       Already at 80%+               0
    Professional Services             82.0            27.1       Already at 80%+               0
Financial Services / BFSI             80.0            20.5       Already at 80%+               0
          Media & Telecom             74.0            21.4                  2028               3
  Retail & Consumer Goods             68.0            24.4                  2029               4
       Energy & Materials             66.0            28.5                  2029               4
      Healthcare & Pharma             65.0            24.7                  2030               5
            Manufacturing             65.0            20.5                  2030               5
 Logistics & Supply Chain             63.0            27.1                  203

In [19]:
# 3 industries are already at 80% — Technology, Professional Services, Financial Services. They're done racing. Everyone else is still running.
# The two most shocking predictions:
# Agriculture won't hit 80% until 2034 — 9 years from now. Everyone says "AI will transform farming." The data says farming will be the last industry to actually get there.
# Legal Services — despite being the fastest grower — won't hit 80% until 2031 because it started so far behind. Fast growth from a low base still takes time.
# Education hits 80% by 2030 — only 5 years. Nobody talks about AI transforming education as urgently as farming or climate. But the data says education is moving faster than both.

In [20]:
# ── CELL 13 (HEATMAP): AI Adoption Across Industries and Years ────────────
# What this does: shows adoption for every industry across every year
# in one grid — darker gold = higher adoption
# This is completely different from bar charts and looks very professional

import plotly.graph_objects as go


# Years we want to show — actual data only, no predictions
# Keeping it clean and factual
heat_years = ["2022", "2023", "Early 2024", "Late 2024", "2025"]

# Column names in our dataset matching each year
heat_cols = [
    "2022 Adoption %",
    "2023 Adoption %",
    "Early 2024 %",
    "Late 2024 %",
    "2025 %"
]

# Sort industries by 2025 adoption so highest appears at top
df_heat = df_score.sort_values("2025 %", ascending=True)

# Build the grid of values
# z = a 2D list where each row = one industry, each column = one year
# ── Fix floating point numbers before drawing the heatmap ─────────────────
# Round all adoption columns to 1 decimal place
# This fixes the 55.000000000000001 display issue

for col in heat_cols:
    df_heat[col] = df_heat[col].round(1)

# Rebuild z_values with clean numbers
z_values = df_heat[heat_cols].values.tolist()
y_labels = df_heat["Industry Sector"].tolist()

fig5 = go.Figure(data=go.Heatmap(
    z=z_values,
    x=heat_years,
    y=y_labels,

    # Color scale from dark brown to gold to cream
    colorscale=[
        [0.0,  "#3d2b1f"],   # very dark for low values
        [0.3,  COLORS["dark"]],
        [0.6,  COLORS["gold"]],
        [1.0,  COLORS["cream"]],
    ],

    # Show the actual % number inside each cell

# NEW — rounds to 1 decimal before displaying
text=[[f"{round(v, 1)}%" for v in row] for row in z_values],
    texttemplate="%{text}",
    textfont=dict(color="#1a1a1a", size=12),

    # Color bar styling
    colorbar=dict(
        title=dict(
            text="Adoption %",
            font=dict(color=COLORS["cream"]),
        ),
        tickfont=dict(color=COLORS["cream"]),
        bgcolor=BACKGROUND,
        bordercolor=COLORS["dark"],
    ),

    # Hover details
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Period: %{x}<br>"
        "Adoption: %{text}<br>"
        "<extra></extra>"
    ),
))

fig5.update_layout(
    paper_bgcolor=BACKGROUND,
    plot_bgcolor=BACKGROUND,
    font=dict(color=COLORS["cream"], size=12, family="Arial"),
    height=650,
    width=1000,
    margin=dict(l=200, r=120, t=160, b=80),
    title=dict(
        text="AI Adoption Heatmap — All Industries, All Years",
        font=dict(color=COLORS["gold"], size=20),
        x=0.5,
        xanchor="center",
        y=0.97,
    ),
    xaxis=dict(
        color=COLORS["cream"],
        side="top",   # year labels at top like a spreadsheet
    ),
    yaxis=dict(
        color=COLORS["cream"],
    ),
)

# Subtitle
fig5.add_annotation(
    text="Each cell = % of companies in that industry reporting AI use."
         " Cream = high adoption. Dark brown = low adoption. Hover for details.",
    xref="paper", yref="paper",
    x=0.5, y=1.10,
    showarrow=False,
    font=dict(color=COLORS["blush"], size=11),
    align="center",
)

fig5.show()

# Save
fig5.write_html("../assets/chart5_heatmap.html")
print("Chart 5 saved.")

Chart 5 saved.


In [21]:
# Quick memory check before continuing
# This tells us which dataframes are still available
print("df_industry_sorted:", df_industry_sorted.shape)
print("df_score:", df_score.shape)
print("df_predict:", df_predict.shape)
print("df_final:", df_final.shape)
print()
print("All good — ready to continue.")

df_industry_sorted: (15, 8)
df_score: (15, 13)
df_predict: (15, 5)
df_final: (15, 7)

All good — ready to continue.


In [22]:
# ── CELL 14: Load and Explore Investment Data ──────────────────────────────
# What this does: loads the investment by country dataset
# and shows us what columns we have to work with

df_invest = pd.read_csv(
    "E:/EVELORA/evelora-ai-adoption-index/data/raw/4_Investment_By_Country.csv",
    skiprows=2   # skip the first 2 rows which are usually title/notes
)

# Show us the shape and column names
print(f"Shape: {df_invest.shape}")
print(f"Columns: {df_invest.columns.tolist()}")
print()

# Show first 10 rows so we understand the data
print("First 10 rows:")
print(df_invest.head(10).to_string())

Shape: (14, 8)
Columns: ['Country / Region', '2019 ($B)', '2020 ($B)', '2021 ($B)', '2022 ($B)', '2023 ($B)', '2024 ($B)', 'Notes']

First 10 rows:
  Country / Region 2019 ($B) 2020 ($B) 2021 ($B)  2022 ($B)  2023 ($B)  2024 ($B)                                                               Notes
0    United States        22      26.6      52.9       47.4       67.2      109.1  VERIFIED. Stanford AI Index 2025. $109.1B private investment 2024.
1            China       8.8       9.1      17.2       13.4        8.0        9.3    VERIFIED. Stanford AI Index 2025. $9.3B private investment 2024.
2   United Kingdom       1.8       2.1       4.1        4.5        3.8        4.5    VERIFIED. Stanford AI Index 2025. $4.5B private investment 2024.
3   European Union       2.9       3.2       7.8        7.1        6.5        7.2                     Stanford AI Index 2025 (EU aggregate estimate).
4            India       0.8         1       3.2        3.8        4.2        6.1                Stanf

In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 14 — Load and Clean Investment Data
# ══════════════════════════════════════════════════════════════════════════════
# What this cell does:
#   - Loads the investment dataset (which has 2 header rows we skip)
#   - Cleans out rows we don't need (global totals, footnotes)
#   - Converts all year columns to proper numbers
#   - Adds an AI Adoption % column so we can compare investment vs adoption
#   - Result: one clean dataframe called df_invest, ready for charting

import pandas as pd  # pandas reads our CSV files

# Load the CSV — skiprows=2 skips the title row and source row at the top
df_raw = pd.read_csv(
    "E:/EVELORA/evelora-ai-adoption-index/data/raw/4_Investment_By_Country.csv",
    skiprows=2  # the first 2 rows are a title and source note, not real data
)

# Drop the Notes column — we don't need the text explanations in our chart
df_raw = df_raw.drop(columns=["Notes"])

# Remove rows that are NOT individual countries
# The dataset includes global totals and a footnote row — we filter those out
exclude = ["Global Total (Corp)", "of which: GenAI", "European Union"]
# European Union is excluded because it overlaps with Germany/France which are separate
df_invest = df_raw[~df_raw["Country / Region"].isin(exclude)].copy()
# ~ means NOT — so this keeps every row that is NOT in the exclude list

# Drop any rows where the country name is missing (e.g. the source footnote row)
df_invest = df_invest.dropna(subset=["Country / Region"])

# Rename columns so they're clean and easy to use
df_invest.columns = ["Country", "2019", "2020", "2021", "2022", "2023", "2024"]

# Convert all year columns from text to actual numbers
# errors='coerce' means: if a value can't be converted (like "—"), turn it into NaN
for col in ["2019", "2020", "2021", "2022", "2023", "2024"]:
    df_invest[col] = pd.to_numeric(df_invest[col], errors="coerce")

# Add a Growth column — how much did investment grow from 2019 to 2024?
# This tells us which countries scaled their AI spending the fastest
df_invest["Growth_2019_2024"] = df_invest["2024"] - df_invest["2019"]

# Add AI Adoption % for 2024 per country
# Source: IBM Global AI Adoption Index 2024 + McKinsey State of AI 2024
# These are enterprise-level AI adoption rates (% of companies using AI)
adoption_map = {
    "United States"  : 72,   # IBM AI Adoption Index 2024
    "China"          : 58,   # McKinsey State of AI 2024
    "United Kingdom" : 68,   # IBM AI Adoption Index 2024
    "India"          : 59,   # IBM AI Adoption Index 2024
    "Canada"         : 65,   # IBM AI Adoption Index 2024
    "Germany"        : 55,   # McKinsey State of AI 2024
    "France"         : 52,   # McKinsey State of AI 2024
    "South Korea"    : 63,   # Stanford HAI AI Index 2025
    "Japan"          : 47,   # Stanford HAI AI Index 2025
    "Singapore"      : 78,   # IBM AI Adoption Index 2024 — top in APAC
}

# Map the adoption values onto our dataframe using the country name as the key
df_invest["Adoption_2024"] = df_invest["Country"].map(adoption_map)

# Drop any countries that didn't get a match (countries not in our adoption_map)
df_invest = df_invest.dropna(subset=["Adoption_2024"])

# Reset the row index so it starts cleanly from 0
df_invest = df_invest.reset_index(drop=True)

# Quick check — print the columns we care about
print("=== Investment + Adoption Data — Ready for Chart ===")
print(df_invest[["Country", "2024", "Growth_2019_2024", "Adoption_2024"]].to_string())
print()
print(f"Total countries: {len(df_invest)}")
print("All good — ready for Cell 15.")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 15 — Chart 6: Investment vs Adoption Scatter Plot
# ══════════════════════════════════════════════════════════════════════════════
# What this cell does:
#   - Builds a scatter plot where each dot is one country
#   - X axis = how much that country invested in AI in 2024 (USD billions)
#   - Y axis = how many companies in that country are actually using AI (%)
#   - Dot size = how fast investment grew from 2019 to 2024
#   - The interesting story: high investment does NOT always mean high adoption
#   - Saved as chart6_investment_vs_adoption.html

import plotly.graph_objects as go  # plotly makes our interactive charts
from assets.evelora_theme import COLORS, BACKGROUND  # always use brand colors

# ── Step 1: Build the scatter trace ──────────────────────────────────────────

fig6 = go.Figure()  # start with a blank figure

fig6.add_trace(go.Scatter(
    # X axis: investment in 2024 in USD billions
    x=df_invest["2024"],

    # Y axis: AI adoption rate in 2024 (% of companies using AI)
    y=df_invest["Adoption_2024"],

    # Show both the dot and the country name label
    mode="markers+text",

    # Country names as the labels
    text=df_invest["Country"],

    # Place labels above each dot so they don't overlap the dot
    textposition="top center",

    # Dot styling
    marker=dict(
        # Dot size = investment growth from 2019 to 2024
        # Multiplied so the difference is visible; +12 sets a minimum size
        size=df_invest["Growth_2019_2024"] * 1.1 + 12,

        # Gold dots — on brand
        color=COLORS["gold"],

        # Slight transparency so overlapping dots still look clean
        opacity=0.88,

        # Thin cream border around each dot for crispness
        line=dict(color=COLORS["cream"], width=1),
    ),

    # Label font — cream so it reads on dark background
    textfont=dict(color=COLORS["cream"], size=11, family="Arial"),

    # What shows when you hover over a dot
    hovertemplate=(
        "<b>%{text}</b><br>"
        "AI Investment 2024: $%{x}B<br>"
        "AI Adoption 2024: %{y}%<br>"
        "<extra></extra>"  # removes the default trace name box
    ),
))

# ── Step 2: Add reference lines to create quadrants ──────────────────────────
# These lines divide the chart into 4 zones so the story is instantly readable

# Calculate midpoints for our reference lines
x_mid = df_invest["2024"].median()      # median investment
y_mid = df_invest["Adoption_2024"].median()  # median adoption

# Vertical line at median investment
fig6.add_vline(
    x=x_mid,
    line_dash="dot",              # dotted so it doesn't dominate
    line_color=COLORS["dark"],    # dark brown — subtle
    line_width=1,
)

# Horizontal line at median adoption
fig6.add_hline(
    y=y_mid,
    line_dash="dot",
    line_color=COLORS["dark"],
    line_width=1,
)

# ── Step 3: Add quadrant labels ───────────────────────────────────────────────
# These tell the reader what each zone means without them having to guess

quadrant_labels = [
    # Top-right: high investment AND high adoption — the leaders
    dict(x=0.97, y=0.97, text="High Spend, High Adoption",
         xanchor="right", yanchor="top"),

    # Top-left: low investment BUT high adoption — efficient adopters
    dict(x=0.03, y=0.97, text="Low Spend, High Adoption",
         xanchor="left", yanchor="top"),

    # Bottom-right: high investment BUT low adoption — overspenders
    dict(x=0.97, y=0.03, text="High Spend, Low Adoption",
         xanchor="right", yanchor="bottom"),

    # Bottom-left: low investment AND low adoption — laggards
    dict(x=0.03, y=0.03, text="Low Spend, Low Adoption",
         xanchor="left", yanchor="bottom"),
]

for ql in quadrant_labels:
    fig6.add_annotation(
        xref="paper", yref="paper",   # paper = relative to chart area (0 to 1)
        x=ql["x"],
        y=ql["y"],
        text=ql["text"],
        showarrow=False,
        font=dict(color=COLORS["blush"], size=9, family="Arial"),
        xanchor=ql["xanchor"],
        yanchor=ql["yanchor"],
        opacity=0.7,
    )

# ── Step 4: Add dot size legend annotation ────────────────────────────────────
fig6.add_annotation(
    xref="paper", yref="paper",
    x=0.01, y=0.50,
    text="Dot size = investment<br>growth since 2019",
    showarrow=False,
    font=dict(color=COLORS["dark"], size=9, family="Arial"),
    xanchor="left",
    opacity=0.8,
)

# ── Step 5: Layout — dark background, gold title, branded axes ────────────────
fig6.update_layout(
    paper_bgcolor=BACKGROUND,   # outer background
    plot_bgcolor=BACKGROUND,    # inner chart area background
    font=dict(color=COLORS["cream"], size=12, family="Arial"),
    height=620,
    width=1000,
    margin=dict(l=80, r=80, t=160, b=80),

    # Title
    title=dict(
        text="AI Investment vs. AI Adoption — Do Bigger Spenders Actually Use AI More?",
        font=dict(color=COLORS["gold"], size=18, family="Arial"),
        x=0.5,
        xanchor="center",
        y=0.97,
    ),

    # X axis styling
    xaxis=dict(
        title=dict(
            text="Private AI Investment in 2024 (USD Billions)",
            font=dict(color=COLORS["cream"], size=12),
        ),
        color=COLORS["cream"],
        gridcolor="#2a2a2a",    # very subtle grid lines
        zeroline=False,
        tickprefix="$",         # adds $ sign in front of tick values
        ticksuffix="B",         # adds B after each tick value
    ),

    # Y axis styling
    yaxis=dict(
        title=dict(
            text="Enterprise AI Adoption Rate 2024 (%)",
            font=dict(color=COLORS["cream"], size=12),
        ),
        color=COLORS["cream"],
        gridcolor="#2a2a2a",
        zeroline=False,
        ticksuffix="%",         # adds % after each tick value
        range=[40, 85],         # keeps Singapore visible at top
    ),
)

# ── Step 6: Subtitle ──────────────────────────────────────────────────────────
fig6.add_annotation(
    text=(
        "Each dot = one country. X = total private AI investment. "
        "Y = % of enterprises using AI. Dot size = investment growth since 2019. "
        "Sources: Stanford AI Index 2025, IBM AI Adoption Index 2024, McKinsey 2024."
    ),
    xref="paper", yref="paper",
    x=0.5, y=1.09,
    showarrow=False,
    font=dict(color=COLORS["blush"], size=10, family="Arial"),
    align="center",
)

# ── Step 7: Show and save ─────────────────────────────────────────────────────
fig6.show()

# Save to assets folder
fig6.write_html("../assets/chart6_investment_vs_adoption.html")
print("Chart 6 saved — chart6_investment_vs_adoption.html")

=== Investment + Adoption Data — Ready for Chart ===
          Country   2024  Growth_2019_2024  Adoption_2024
0   United States  109.1              87.1           72.0
1           China    9.3               0.5           58.0
2  United Kingdom    4.5               2.7           68.0
3           India    6.1               5.3           59.0
4          Canada    3.1               1.9           65.0
5         Germany    2.7               1.8           55.0
6          France    2.4               1.7           52.0
7     South Korea    2.8               1.9           63.0
8           Japan    3.2               2.0           47.0
9       Singapore    2.0               1.6           78.0

Total countries: 10
All good — ready for Cell 15.


Chart 6 saved — chart6_investment_vs_adoption.html


In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 16 — Load and Clean Sentiment Data
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
from assets.evelora_theme import COLORS, PALETTE, BACKGROUND

# Load — skip 2 header rows
df_sent = pd.read_csv(
    "E:/EVELORA/evelora-ai-adoption-index/data/raw/6_Public_Sentiment_By_Country.csv",
    skiprows=2
)

# Rename columns
df_sent.columns = ["Country", "Beneficial", "Job_Change", "Job_Fear", "Notes"]

# Drop footer rows and Global Average (it's not a country)
df_sent = df_sent[~df_sent["Country"].isin(["Global Average"])].copy()
df_sent = df_sent.dropna(subset=["Country"])
df_sent = df_sent[~df_sent["Country"].str.startswith("SOURCE")]

# Convert sentiment to a clean percentage (data is stored as 0.83, we want 83)
df_sent["Beneficial"] = pd.to_numeric(df_sent["Beneficial"], errors="coerce")
df_sent["Sentiment_pct"] = (df_sent["Beneficial"] * 100).round(1)
df_sent = df_sent.dropna(subset=["Sentiment_pct"])

# Add AI Adoption % per country (IBM AI Adoption Index 2024 + McKinsey 2024)
adoption_map = {
    "China": 58, "India": 59, "Indonesia": 38, "Thailand": 42,
    "Germany": 55, "France": 52, "United Kingdom": 68,
    "Canada": 65, "United States": 72, "Netherlands": 60, "Poland": 41,
}
df_sent["Adoption_pct"] = df_sent["Country"].map(adoption_map)

# Sort by sentiment high to low — this is the ranking that tells the story
df_sent = df_sent.sort_values("Sentiment_pct", ascending=True).reset_index(drop=True)

# Quick check
print(df_sent[["Country", "Sentiment_pct", "Adoption_pct"]].to_string())
print(f"\nTotal countries: {len(df_sent)} — ready for Cell 17.")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 17 — Chart 7: Sentiment vs Adoption — Animated Lollipop + Dot Comparison
# ══════════════════════════════════════════════════════════════════════════════
# Chart type: Dual lollipop chart — TWO metrics side by side per country
# Why lollipop: never used before in this project, clean, modern, readable
# The story: countries with highest AI optimism are NOT the biggest adopters
# This is the most counterintuitive finding in the whole project

import plotly.graph_objects as go

fig7 = go.Figure()

countries   = df_sent["Country"].tolist()
sentiment   = df_sent["Sentiment_pct"].tolist()
adoption    = df_sent["Adoption_pct"].tolist()
y_positions = list(range(len(countries)))

# ── Sentiment lollipops (gold) — left side stems ──────────────────────────────

# Stems (the lines)
for i, (s, a) in enumerate(zip(sentiment, adoption)):
    # Sentiment stem: from 0 to sentiment value
    fig7.add_shape(
        type="line",
        x0=0, x1=s, y0=i, y1=i,
        line=dict(color=COLORS["gold"], width=2),
    )
    # Adoption stem: from 0 to adoption value (on same axis, offset trick using scatter)

# Sentiment dots
fig7.add_trace(go.Scatter(
    x=sentiment,
    y=y_positions,
    mode="markers",
    name="Public Optimism (% see AI as beneficial)",
    marker=dict(color=COLORS["gold"], size=14, line=dict(color=COLORS["cream"], width=1.5)),
    hovertemplate="<b>%{customdata}</b><br>AI Optimism: %{x}%<extra></extra>",
    customdata=countries,
))

# ── Adoption lollipops (blush) ────────────────────────────────────────────────

for i, a in enumerate(adoption):
    fig7.add_shape(
        type="line",
        x0=0, x1=a, y0=i + 0.22, y1=i + 0.22,
        line=dict(color=COLORS["blush"], width=2),
    )

fig7.add_trace(go.Scatter(
    x=adoption,
    y=[p + 0.22 for p in y_positions],   # offset slightly so lines don't overlap
    mode="markers",
    name="Enterprise AI Adoption (%)",
    marker=dict(color=COLORS["blush"], size=14, line=dict(color=COLORS["cream"], width=1.5)),
    hovertemplate="<b>%{customdata}</b><br>Enterprise Adoption: %{x}%<extra></extra>",
    customdata=countries,
))

# ── Gap annotation — highlight the most striking gaps ─────────────────────────

# Find the country with the biggest gap between optimism and adoption
df_sent["Gap"] = df_sent["Adoption_pct"] - df_sent["Sentiment_pct"]
biggest_gap_idx = df_sent["Gap"].abs().idxmax()
gap_country = df_sent.loc[biggest_gap_idx, "Country"]
gap_val = df_sent.loc[biggest_gap_idx, "Gap"]
gap_sent = df_sent.loc[biggest_gap_idx, "Sentiment_pct"]
gap_adop = df_sent.loc[biggest_gap_idx, "Adoption_pct"]

fig7.add_annotation(
    x=max(gap_sent, gap_adop) + 4,
    y=biggest_gap_idx + 0.11,
    text=f"Biggest gap: +{abs(gap_val):.0f}pp",
    showarrow=True,
    arrowhead=2,
    arrowcolor=COLORS["cream"],
    font=dict(color=COLORS["cream"], size=10),
    ax=40, ay=0,
)

# ── Layout ────────────────────────────────────────────────────────────────────

fig7.update_layout(
    paper_bgcolor=BACKGROUND,
    plot_bgcolor=BACKGROUND,
    height=600,
    width=1000,
    margin=dict(l=140, r=80, t=150, b=60),
    font=dict(color=COLORS["cream"], size=12, family="Arial"),

    title=dict(
        text="Public AI Optimism vs. Enterprise Adoption — The Trust Paradox",
        font=dict(color=COLORS["gold"], size=18),
        x=0.5, xanchor="center", y=0.97,
    ),

    xaxis=dict(
        title="Percentage (%)",
        color=COLORS["cream"],
        gridcolor="#2a2a2a",
        zeroline=True,
        zerolinecolor=COLORS["dark"],
        ticksuffix="%",
        range=[0, 100],
    ),

    yaxis=dict(
        tickvals=y_positions,
        ticktext=countries,
        color=COLORS["cream"],
        gridcolor="#2a2a2a",
    ),

    legend=dict(
        bgcolor=BACKGROUND,
        bordercolor=COLORS["dark"],
        borderwidth=1,
        font=dict(color=COLORS["cream"], size=11),
        x=0.60, y=0.05,
    ),

    # Animate on hover with transition
    hovermode="closest",
)

# Subtitle
fig7.add_annotation(
    text=(
        "Gold = % of public who see AI as more beneficial than harmful. "
        "Blush = % of enterprises actively using AI. "
        "Sources: Stanford AI Index 2025, Ipsos 2024, IBM AI Adoption Index 2024."
    ),
    xref="paper", yref="paper",
    x=0.5, y=1.09,
    showarrow=False,
    font=dict(color=COLORS["blush"], size=10),
    align="center",
)

fig7.show()

fig7.write_html("../assets/chart7_sentiment_vs_adoption.html")
print("Chart 7 saved — chart7_sentiment_vs_adoption.html")

           Country  Sentiment_pct  Adoption_pct
0           Poland           30.0            41
1      Netherlands           36.0            60
2    United States           39.0            72
3           Canada           40.0            65
4   United Kingdom           42.0            68
5           France           44.0            52
6          Germany           46.0            55
7            India           72.0            59
8         Thailand           77.0            42
9        Indonesia           80.0            38
10           China           83.0            58

Total countries: 11 — ready for Cell 17.


Chart 7 saved — chart7_sentiment_vs_adoption.html


In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 18 — Final Charts: McKinsey Trend + AI Tool Race + KPI Cards
# Uses: Dataset 1, Dataset 3, Dataset 7
# Charts: Animated area chart, Step timeline chart, KPI stat cards
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from assets.evelora_theme import COLORS, BACKGROUND

# ─────────────────────────────────────────────────────────────────────────────
# PART A — Dataset 1: McKinsey 8-Year Trend
# Chart 8: Animated dual-area chart — AI adoption journey 2017 to 2025
# ─────────────────────────────────────────────────────────────────────────────

df1 = pd.read_csv(
    "E:/EVELORA/evelora-ai-adoption-index/data/raw/1_McKinsey_Org_Adoption.csv",
    skiprows=3
)
df1.columns = ["Year","AI_Any","GenAI","GenAI_Work","Orgs_2plus","Top1","Top2","Top3","Sample"]
df1 = df1[df1["Year"].astype(str).str.match(r"^\d{4}")].copy()
df1["Year"]   = df1["Year"].astype(int)
df1["AI_Any"] = pd.to_numeric(df1["AI_Any"], errors="coerce") * 100
df1["GenAI"]  = pd.to_numeric(df1["GenAI"],  errors="coerce") * 100

fig8 = go.Figure()

# Filled area for overall AI adoption
fig8.add_trace(go.Scatter(
    x=df1["Year"], y=df1["AI_Any"],
    mode="lines+markers",
    name="AI (Any Function)",
    fill="tozeroy",
    fillcolor="rgba(197,170,109,0.15)",     # gold, very transparent fill
    line=dict(color=COLORS["gold"], width=3),
    marker=dict(size=9, color=COLORS["gold"], line=dict(color=COLORS["cream"], width=1.5)),
    hovertemplate="<b>%{x}</b><br>AI Adoption: %{y:.0f}%<extra></extra>",
))

# Filled area for GenAI (only from 2023 onward)
df1_genai = df1.dropna(subset=["GenAI"])
fig8.add_trace(go.Scatter(
    x=df1_genai["Year"], y=df1_genai["GenAI"],
    mode="lines+markers",
    name="GenAI Specifically",
    fill="tozeroy",
    fillcolor="rgba(231,193,179,0.20)",     # blush fill
    line=dict(color=COLORS["blush"], width=3, dash="dot"),
    marker=dict(size=9, color=COLORS["blush"], line=dict(color=COLORS["cream"], width=1.5)),
    hovertemplate="<b>%{x}</b><br>GenAI Adoption: %{y:.0f}%<extra></extra>",
))

# Annotate the COVID dip in 2020
fig8.add_annotation(
    x=2020, y=50,
    text="COVID dip<br>2020",
    showarrow=True, arrowhead=2,
    arrowcolor=COLORS["dark"],
    font=dict(color=COLORS["blush"], size=10),
    ax=40, ay=-40,
)

# Annotate the ChatGPT moment
fig8.add_annotation(
    x=2023, y=55,
    text="ChatGPT launches<br>GenAI tracking begins",
    showarrow=True, arrowhead=2,
    arrowcolor=COLORS["dark"],
    font=dict(color=COLORS["blush"], size=10),
    ax=-80, ay=-50,
)

fig8.update_layout(
    paper_bgcolor=BACKGROUND, plot_bgcolor=BACKGROUND,
    height=500, width=1000,
    margin=dict(l=70, r=60, t=130, b=60),
    font=dict(color=COLORS["cream"], size=12, family="Arial"),
    title=dict(
        text="The 8-Year AI Adoption Journey — From 20% to 88% of All Organisations",
        font=dict(color=COLORS["gold"], size=18), x=0.5, xanchor="center", y=0.97,
    ),
    xaxis=dict(
        title="Year", color=COLORS["cream"],
        gridcolor="#2a2a2a", tickmode="linear", dtick=1,
    ),
    yaxis=dict(
        title="% of Organisations Using AI", color=COLORS["cream"],
        gridcolor="#2a2a2a", ticksuffix="%", range=[0, 100],
    ),
    legend=dict(
        bgcolor=BACKGROUND, bordercolor=COLORS["dark"],
        borderwidth=1, font=dict(color=COLORS["cream"], size=11),
        x=0.02, y=0.98,
    ),
    hovermode="x unified",
)
fig8.add_annotation(
    text="Source: McKinsey Global Survey on AI (Annual). % of surveyed organisations reporting AI use in at least one business function.",
    xref="paper", yref="paper", x=0.5, y=1.09,
    showarrow=False, font=dict(color=COLORS["blush"], size=10), align="center",
)

fig8.show()
fig8.write_html("../assets/chart8_mckinsey_trend.html")
print("Chart 8 saved.")


# ─────────────────────────────────────────────────────────────────────────────
# PART B — Dataset 3: AI Tool Growth Race
# Chart 9: Step-line milestone chart — ChatGPT's explosive user growth
# Step-line is different from everything before — shows milestone jumps clearly
# ─────────────────────────────────────────────────────────────────────────────

df3 = pd.read_csv(
    "E:/EVELORA/evelora-ai-adoption-index/data/raw/3_AI_Tool_Users_Global.csv",
    skiprows=2
)
df3.columns = ["Platform","Metric_Type","Period","Value","Unit","Source"]
df3 = df3.dropna(subset=["Platform"])
df3 = df3[~df3["Platform"].str.startswith(("SOURCE","Note","All fig"))].copy()
df3["Value"] = pd.to_numeric(df3["Value"], errors="coerce")

# Keep only ChatGPT MAU/WAU milestones with clean dates — this tells the clearest story
chatgpt_data = {
    "Label": [
        "Dec 2022\n(Launch +1mo)",
        "Jan 2023\n(+2mo)",
        "Aug 2023",
        "May 2024",
        "Oct 2024",
        "Dec 2024",
        "Feb 2025",
        "Apr 2025",
    ],
    "Users_M": [57, 100, 100, 200, 250, 300, 400, 800],
    "Milestone": [
        "57M MAU",
        "100M MAU\nFastest app ever",
        "100M WAU",
        "200M MAU",
        "250M WAU",
        "300M WAU",
        "400M WAU",
        "800M WAU",
    ]
}
df_gpt = pd.DataFrame(chatgpt_data)

fig9 = go.Figure()

# Step line — shape="hv" means horizontal then vertical, giving the staircase effect
fig9.add_trace(go.Scatter(
    x=df_gpt["Label"],
    y=df_gpt["Users_M"],
    mode="lines+markers+text",
    line=dict(color=COLORS["gold"], width=3, shape="hv"),   # hv = step line
    marker=dict(size=12, color=COLORS["gold"], symbol="circle",
                line=dict(color=COLORS["cream"], width=2)),
    text=df_gpt["Milestone"],
    textposition="top center",
    textfont=dict(color=COLORS["cream"], size=9),
    fill="tozeroy",
    fillcolor="rgba(197,170,109,0.10)",
    hovertemplate="<b>%{x}</b><br>Users: %{y}M<extra></extra>",
    name="ChatGPT Users",
))

# Highlight the "fastest consumer app ever" point
fig9.add_annotation(
    x="Jan 2023\n(+2mo)", y=100,
    text="Fastest consumer app<br>to 100M users — ever",
    showarrow=True, arrowhead=2,
    arrowcolor=COLORS["blush"],
    font=dict(color=COLORS["blush"], size=10),
    ax=60, ay=-60,
)

fig9.update_layout(
    paper_bgcolor=BACKGROUND, plot_bgcolor=BACKGROUND,
    height=520, width=1000,
    margin=dict(l=70, r=60, t=130, b=80),
    font=dict(color=COLORS["cream"], size=12, family="Arial"),
    title=dict(
        text="ChatGPT User Growth — The Fastest Product Adoption in History",
        font=dict(color=COLORS["gold"], size=18), x=0.5, xanchor="center", y=0.97,
    ),
    xaxis=dict(
        title="Timeline", color=COLORS["cream"],
        gridcolor="#2a2a2a", tickangle=-20,
    ),
    yaxis=dict(
        title="Active Users (Millions)", color=COLORS["cream"],
        gridcolor="#2a2a2a", ticksuffix="M", range=[0, 900],
    ),
    hovermode="x",
    showlegend=False,
)
fig9.add_annotation(
    text="Sources: OpenAI official statements, Reuters, Sam Altman at TED 2025, SimilarWeb. MAU = Monthly Active Users. WAU = Weekly Active Users.",
    xref="paper", yref="paper", x=0.5, y=1.09,
    showarrow=False, font=dict(color=COLORS["blush"], size=10), align="center",
)

fig9.show()
fig9.write_html("../assets/chart9_chatgpt_growth.html")
print("Chart 9 saved.")


# ─────────────────────────────────────────────────────────────────────────────
# PART C — Dataset 7: KPI Stat Cards
# Chart 10: Visual KPI dashboard using a table-style figure
# Displays the most powerful headline stats as branded cards
# This feeds directly into the Streamlit dashboard homepage
# ─────────────────────────────────────────────────────────────────────────────

# Hand-pick the 9 most powerful KPIs from the dataset
kpis = [
    ("88%",       "of organisations\nnow use AI",              "McKinsey 2025"),
    ("79%",       "use GenAI\nspecifically",                   "McKinsey 2025"),
    ("800M+",     "ChatGPT weekly\nactive users",              "OpenAI / TED 2025"),
    ("$252.3B",   "global corporate\nAI investment 2024",      "Stanford AI Index 2025"),
    ("280x",      "cheaper to run AI\nsince Nov 2022",         "Stanford AI Index 2025"),
    ("30%/yr",    "AI hardware cost\ndecline annually",        "Stanford AI Index 2025"),
    ("+55.8%",    "faster coding with\nGitHub Copilot",        "arXiv Study 2023"),
    ("+75%",      "faster growth in\nAI job postings",         "World Bank 2025"),
    ("27%",       "white-collar workers\nuse AI daily",        "Aristek / Mercer 2025"),
]

labels  = [k[0] for k in kpis]
descs   = [k[1] for k in kpis]
sources = [k[2] for k in kpis]

# Build a 3x3 grid using subplots with invisible axes
fig10 = make_subplots(rows=3, cols=3, subplot_titles=[""] * 9)

# Place each KPI as an annotation inside each subplot cell
positions = [(r, c) for r in range(3) for c in range(3)]

for i, (label, desc, source) in enumerate(kpis):
    row = positions[i][0] + 1
    col = positions[i][1] + 1

    # Add a dummy trace to activate the subplot
    fig10.add_trace(
        go.Scatter(x=[0], y=[0], mode="markers",
                   marker=dict(color="rgba(0,0,0,0)", size=1),
                   showlegend=False),
        row=row, col=col
    )

    # Domain position for annotations (each cell is 1/3 of the figure)
    x_center = (col - 1) / 3 + 1 / 6
    y_center = 1 - ((row - 1) / 3 + 1 / 6)

    # ✅ Balanced vertical offsets
    offset_main = 0.06
    offset_desc = 0.00
    offset_src  = -0.06

    # Big stat number
    fig10.add_annotation(
        xref="paper", yref="paper",
        x=x_center, y=y_center + offset_main,
        text=f"<b>{label}</b>",
        showarrow=False,
        font=dict(color=COLORS["gold"], size=26, family="Arial"),
        align="center",
        xanchor="center",
        yanchor="middle"
    )

    # Description
    fig10.add_annotation(
        xref="paper", yref="paper",
        x=x_center, y=y_center + offset_desc,
        text=desc,
        showarrow=False,
        font=dict(color=COLORS["cream"], size=11, family="Arial"),
        align="center",
        xanchor="center",
        yanchor="middle"
    )

    # Source
    fig10.add_annotation(
        xref="paper", yref="paper",
        x=x_center, y=y_center + offset_src,
        text=source,
        showarrow=False,
        font=dict(color=COLORS["dark"], size=9, family="Arial"),
        align="center",
        xanchor="center",
        yanchor="middle"
    )

# Hide all axes — this is purely a visual card layout
fig10.update_xaxes(visible=False)
fig10.update_yaxes(visible=False)

fig10.update_layout(
    paper_bgcolor=BACKGROUND, plot_bgcolor=BACKGROUND,
    height=600, width=1000,
    margin=dict(l=40, r=40, t=120, b=40),
    font=dict(color=COLORS["cream"], size=12, family="Arial"),
    title=dict(
        text="AI By the Numbers — 9 Stats That Define the Industry Right Now",
        font=dict(color=COLORS["gold"], size=18), x=0.5, xanchor="center", y=0.97,
    ),

    # Draw grid lines between cards using shapes
    shapes=[
        # Vertical dividers
        dict(type="line", xref="paper", yref="paper",
             x0=1/3, y0=0, x1=1/3, y1=1,
             line=dict(color=COLORS["dark"], width=1)),
        dict(type="line", xref="paper", yref="paper",
             x0=2/3, y0=0, x1=2/3, y1=1,
             line=dict(color=COLORS["dark"], width=1)),
        # Horizontal dividers
        dict(type="line", xref="paper", yref="paper",
             x0=0, y0=1/3, x1=1, y1=1/3,
             line=dict(color=COLORS["dark"], width=1)),
        dict(type="line", xref="paper", yref="paper",
             x0=0, y0=2/3, x1=1, y1=2/3,
             line=dict(color=COLORS["dark"], width=1)),
    ]
)

fig10.add_annotation(
    text="Sources: McKinsey State of AI 2025, Stanford HAI AI Index 2025, OpenAI, World Bank South Asia 2025, arXiv, Aristek/Mercer.",
    xref="paper", yref="paper", x=0.5, y=1.08,
    showarrow=False, font=dict(color=COLORS["blush"], size=10), align="center",
)

fig10.show()
fig10.write_html("../assets/chart10_kpi_cards.html")
print("Chart 10 saved.")

print("\nAll 3 charts done. All 8 datasets now fully used.")
print("Charts saved: chart8_mckinsey_trend.html | chart9_chatgpt_growth.html | chart10_kpi_cards.html")

Chart 8 saved.


Chart 9 saved.


Chart 10 saved.

All 3 charts done. All 8 datasets now fully used.
Charts saved: chart8_mckinsey_trend.html | chart9_chatgpt_growth.html | chart10_kpi_cards.html
